# Subgroup discovery

In [29]:
from __future__ import annotations

import re
import numpy as np
import pandas as pd
import pysubgroup as ps

## Configuracion

In [30]:
# CSV_PATH = "/mnt/homeGPU/mcribilles/tfm/clinical_data/clinical_data_limpios_SD.csv"
CSV_PATH="/mnt/homeGPU/mcribilles/tfm/clinical_data/clinical_data_SD_con_geometria_210.csv"

ID_COL = "patient_id"

TARGET_BINARY = "Complicacion_binaria"
TARGETS_MULTILABEL = ["Neumotórax", "Hemorragia", "Derrame_pleural"]

EXTRA_TARGET_COLS = ["Sin_complicación"]  # no usar como descriptor

NUMERIC_COLS = ["Edad", "tamano_nodulo_mm", "profundidad_min_pleura_mm", "profundidad_centroidal_pleura_mm"]
# NUMERIC_COLS = ["Edad"]


N_BINS = 4
MAX_RULE_DEPTH = 3
TOP_K = 20
BEAM_WIDTH = 50

MIN_POS_COUNT = 3          # para binarias: mínimo número de 1s para dejar la variable
MAX_POS_RATIO = 0.97       # si casi siempre es 1, quitar
CORR_DROP_THRESHOLD = 0.95 # quitar variables binarias casi duplicadas

## Utilidades

In [31]:
def merge_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Unifica columnas duplicadas tipo X y X.1, X.2, etc.
    Deja la base X con el máximo fila a fila y elimina las .1, .2...
    """
    dup_pattern = re.compile(r"^(.*)\.(\d+)$")
    cols = list(df.columns)
    base_to_dups = {}

    for c in cols:
        m = dup_pattern.match(c)
        if m:
            base = m.group(1)
            base_to_dups.setdefault(base, []).append(c)

    for base, dups in base_to_dups.items():
        if base in df.columns:
            candidates = [base] + dups
            tmp = df[candidates].apply(pd.to_numeric, errors="coerce")
            df[base] = np.nanmax(tmp.to_numpy(), axis=1)
            df.drop(columns=dups, inplace=True)

    return df


def coerce_binary_int(df: pd.DataFrame, exclude_cols: list[str]) -> pd.DataFrame:
    """
    Convierte columnas que sean binarias {0,1} a int, rellenando NaN con 0.
    """
    for c in df.columns:
        if c in exclude_cols:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        vals = set(pd.Series(s).dropna().unique().tolist())
        if len(vals) > 0 and vals.issubset({0, 1}):
            df[c] = s.fillna(0).astype(int)
    return df


def pick_descriptor_columns(df: pd.DataFrame, target_cols: list[str]) -> list[str]:
    """
    Quita targets e ID, y filtra columnas constantes o muy raras.
    """
    candidates = [c for c in df.columns if c not in target_cols and c != ID_COL]
    keep = []
    n = len(df)

    for c in candidates:
        s = df[c]

        if s.dtype == "object":
            nun = s.nunique(dropna=True)
            if nun >= 2:
                keep.append(c)
            continue

        x = pd.to_numeric(s, errors="coerce")
        nun = x.nunique(dropna=True)
        if nun < 2:
            continue

        vals = set(x.dropna().unique().tolist())
        if vals.issubset({0, 1}):
            pos = int((x.fillna(0) == 1).sum())
            if pos < MIN_POS_COUNT:
                continue
            if pos / max(1, n) > MAX_POS_RATIO:
                continue

        keep.append(c)

    return keep


def drop_highly_correlated_binaries(df: pd.DataFrame, desc_cols: list[str]) -> list[str]:
    """
    Elimina variables binarias casi duplicadas por correlación absoluta.
    """
    bin_cols = []
    for c in desc_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        vals = set(pd.Series(s).dropna().unique().tolist())
        if len(vals) > 0 and vals.issubset({0, 1}):
            bin_cols.append(c)

    if len(bin_cols) < 2:
        return desc_cols

    X = df[bin_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(float)
    corr = X.corr().abs()
    ones = X.sum(axis=0)

    to_drop = set()
    for i in range(len(bin_cols)):
        for j in range(i + 1, len(bin_cols)):
            ci, cj = bin_cols[i], bin_cols[j]
            if corr.loc[ci, cj] >= CORR_DROP_THRESHOLD:
                drop = ci if ones[ci] < ones[cj] else cj
                to_drop.add(drop)

    return [c for c in desc_cols if c not in to_drop]


def discretize_numeric(df: pd.DataFrame, desc_cols: list[str]) -> tuple[pd.DataFrame, list[str]]:
    """
    Discretiza NUMERIC_COLS en bins y sustituye la columna numérica por col_bin categórica.
    """
    df2 = df.copy()
    new_cols = list(desc_cols)

    for c in NUMERIC_COLS:
        if c not in df2.columns or c not in new_cols:
            continue

        x = pd.to_numeric(df2[c], errors="coerce")
        if x.isna().all():
            continue

        try:
            bins = pd.qcut(x, q=N_BINS, duplicates="drop")
        except Exception:
            bins = pd.cut(x, bins=min(N_BINS, x.nunique(dropna=True)))

        bcol = f"{c}_bin"
        df2[bcol] = bins.astype(str).fillna("MISSING")
        df2[bcol] = pd.Categorical(df2[bcol])

        new_cols.remove(c)
        new_cols.append(bcol)

    return df2, new_cols


def build_search_space(df: pd.DataFrame, desc_cols: list[str]) -> list[ps.SelectorBase]:
    """
    Construye selectores:
    • categóricas: col == valor
    • binarias: col == 1
    """
    selectors = []
    for c in desc_cols:
        s = df[c]

        if str(s.dtype) == "category" or s.dtype == "object":
            vals = pd.Series(s.astype(str)).fillna("MISSING").unique().tolist()
            for v in vals:
                selectors.append(ps.EqualitySelector(c, v))
            continue

        x = pd.to_numeric(s, errors="coerce").fillna(0)
        vals = set(x.unique().tolist())
        if vals.issubset({0, 1}):
            selectors.append(ps.EqualitySelector(c, 1))
        else:
            # por si queda algo discreto no binario
            for v in sorted(list(vals)):
                selectors.append(ps.EqualitySelector(c, v))

    return selectors


def auto_min_support(df: pd.DataFrame, target_col: str, base: int = 5, frac: float = 0.05) -> int:
    """
    Heurística para soporte mínimo:
    • al menos 5
    • al menos 5% del dataset
    • y no más que los positivos totales (para que exista)
    """
    n = len(df)
    ms = max(base, int(np.ceil(frac * n)))
    pos = int(pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int).sum())
    return max(2, min(ms, max(2, pos)))  # evita valores absurdos


# def run_sd(df: pd.DataFrame, desc_cols: list[str], target_col: str, out_csv: str) -> pd.DataFrame:
#     """
#     Ejecuta SD para target binario target_col == 1.
#     """
#     ms = auto_min_support(df, target_col)

#     target = ps.BinaryTarget(target_col, 1)
#     search_space = build_search_space(df, desc_cols)

#     task = ps.SubgroupDiscoveryTask(
#         data=df,
#         target=target,
#         search_space=search_space,
#         qf=ps.WRAccQF(),
#         result_set_size=TOP_K,
#         depth=MAX_RULE_DEPTH,
#         constraints=[ps.MinSupportConstraint(ms)],
#     )

#     algo = ps.BeamSearch(beam_width=BEAM_WIDTH)
#     result = algo.execute(task)

#     stats = [
#         "size_sg", "size_dataset",
#         "positives_sg", "positives_dataset",
#         "coverage_sg",
#         "target_share_sg", "target_share_dataset",
#         "lift",
#     ]

#     res_df = result.to_dataframe(statistics_to_show=stats, autoround=True, include_target=True)
#     res_df.to_csv(out_csv, index=False)
#     return res_df

def run_sd(df: pd.DataFrame, desc_cols: list[str], target_col: str, out_csv: str) -> pd.DataFrame:
    ms = auto_min_support(df, target_col)

    target = ps.BinaryTarget(target_col, 1)
    search_space = build_search_space(df, desc_cols)

    task = ps.SubgroupDiscoveryTask(
        data=df,
        target=target,
        search_space=search_space,
        qf=ps.WRAccQF(),
        result_set_size=TOP_K,
        depth=MAX_RULE_DEPTH,
        constraints=[ps.MinSupportConstraint(ms)],
    )

    algo = ps.BeamSearch(beam_width=BEAM_WIDTH)
    result = algo.execute(task)

    stats = [
        "size_sg", "size_dataset",
        "positives_sg", "positives_dataset",
        "coverage_sg",  # OJO: esto es positives_sg/positives_dataset
        "target_share_sg", "target_share_dataset",
        "lift",
    ]

    res_df = result.to_dataframe(statistics_to_show=stats, autoround=True, include_target=True)

    # Renombrar para evitar confusión
    if "coverage_sg" in res_df.columns:
        res_df = res_df.rename(columns={"coverage_sg": "pos_coverage"})  # recall de positivos

    # Añadir soporte real (lo que tú esperabas por "coverage")
    res_df["support_sg"] = (res_df["size_sg"] / res_df["size_dataset"]).round(3)

    # Sanity check opcional:
    # res_df["pos_coverage_check"] = (res_df["positives_sg"] / res_df["positives_dataset"]).round(3)

    res_df.to_csv(out_csv, index=False)
    return res_df

In [32]:
#leemos el csv
df = pd.read_csv(CSV_PATH)

In [34]:
def run_for_target(target_col: str, exclude_comp_cols: list[str], suffix: str):
    # Excluye targets y columnas de complicación del espacio de descriptores
    target_cols = [target_col] + exclude_comp_cols
    target_cols = [c for c in target_cols if c in df.columns]

    desc_cols = pick_descriptor_columns(df, target_cols=target_cols)
    desc_cols = drop_highly_correlated_binaries(df, desc_cols)

    df2, desc_cols2 = discretize_numeric(df, desc_cols)

    print(f"\nTarget: {target_col}")
    print(f"Filas: {len(df2)}")
    print(f"Descriptores finales: {len(desc_cols2)}")
    print("Ejemplo descriptores:", desc_cols2[:20])

    out = f"sd_{target_col}_{suffix}_geometry.csv"
    res = run_sd(df2, desc_cols2, target_col, out_csv=out)

    # Cobertura correcta (por si coverage_sg viene raro)
    if "size_sg" in res.columns and "size_dataset" in res.columns:
        res["coverage_calc"] = (res["size_sg"] / res["size_dataset"]).round(3)

    print(f"Guardado: {out}")
    print(res.head(10))
    return res

In [35]:
#preprocesamiento

# Vacíos a NaN
df = df.replace(r"^\s*$", np.nan, regex=True)

# Duplicados tipo X.1
df = merge_duplicate_columns(df)

# Asegurar que targets son 0/1 int donde aplique
all_comp_cols = TARGETS_MULTILABEL + EXTRA_TARGET_COLS + [TARGET_BINARY]
all_comp_cols = [c for c in all_comp_cols if c in df.columns]
df = coerce_binary_int(df, exclude_cols=[ID_COL] + NUMERIC_COLS)

In [36]:
# 1) SD binario: excluir todas las etiquetas de complicación
exclude_for_binary = TARGETS_MULTILABEL + EXTRA_TARGET_COLS
exclude_for_binary = [c for c in exclude_for_binary if c in df.columns]

if TARGET_BINARY in df.columns:
    run_for_target(TARGET_BINARY, exclude_for_binary, suffix="bin")


Target: Complicacion_binaria
Filas: 210
Descriptores finales: 17
Ejemplo descriptores: ['AOS', 'EPOC', 'Hipertensión_pulmonar', 'Obesidad', 'Bronquiectasias', 'Sexo_binaria', 'Fibrosis_any', 'Dislipemia_any', 'DM_any', 'Tabac_any', 'Alcohol_any', 'Enfisema_any', 'CardioRisk', 'Edad_bin', 'tamano_nodulo_mm_bin', 'profundidad_min_pleura_mm_bin', 'profundidad_centroidal_pleura_mm_bin']
Guardado: sd_Complicacion_binaria_bin_geometry.csv
   quality                                           subgroup  \
0    0.036  Sexo_binaria==1 AND tamano_nodulo_mm_bin=='(17...   
1    0.036  Tabac_any==1 AND tamano_nodulo_mm_bin=='(17.90...   
2    0.036                                    Enfisema_any==1   
3    0.033           tamano_nodulo_mm_bin=='(17.908, 27.847]'   
4    0.031                Enfisema_any==1 AND Sexo_binaria==1   
5    0.031                   Enfisema_any==1 AND Tabac_any==1   
6    0.029  Sexo_binaria==1 AND Tabac_any==1 AND tamano_no...   
7    0.029  profundidad_centroidal_pleura_

/mnt/homeGPU/mcribilles/conda_envs/sd/lib/python3.11/site-packages/pysubgroup/binary_target.py:356: RuntimeWarning: invalid value encountered in divide
  p_subgroup = np.divide(positives_subgroup, instances_subgroup)


In [37]:
# 2) SD por tipo de complicación (one-vs-rest)
# Para cada etiqueta, excluimos:
# - Complicacion_binaria
# - Sin_complicación
# - el resto de etiquetas de complicación
for lab in TARGETS_MULTILABEL:
    if lab not in df.columns:
        continue
    exclude_for_label = [TARGET_BINARY] + EXTRA_TARGET_COLS + [c for c in TARGETS_MULTILABEL if c != lab]
    exclude_for_label = [c for c in exclude_for_label if c in df.columns]
    run_for_target(lab, exclude_for_label, suffix="multilabel")


Target: Neumotórax
Filas: 210
Descriptores finales: 17
Ejemplo descriptores: ['AOS', 'EPOC', 'Hipertensión_pulmonar', 'Obesidad', 'Bronquiectasias', 'Sexo_binaria', 'Fibrosis_any', 'Dislipemia_any', 'DM_any', 'Tabac_any', 'Alcohol_any', 'Enfisema_any', 'CardioRisk', 'Edad_bin', 'tamano_nodulo_mm_bin', 'profundidad_min_pleura_mm_bin', 'profundidad_centroidal_pleura_mm_bin']


/mnt/homeGPU/mcribilles/conda_envs/sd/lib/python3.11/site-packages/pysubgroup/binary_target.py:356: RuntimeWarning: invalid value encountered in divide
  p_subgroup = np.divide(positives_subgroup, instances_subgroup)
/mnt/homeGPU/mcribilles/conda_envs/sd/lib/python3.11/site-packages/pysubgroup/binary_target.py:356: RuntimeWarning: invalid value encountered in divide
  p_subgroup = np.divide(positives_subgroup, instances_subgroup)


Guardado: sd_Neumotórax_multilabel_geometry.csv
   quality                                           subgroup  \
0    0.041  Sexo_binaria==1 AND tamano_nodulo_mm_bin=='(17...   
1    0.037           tamano_nodulo_mm_bin=='(17.908, 27.847]'   
2    0.037  Sexo_binaria==1 AND Tabac_any==1 AND tamano_no...   
3    0.035  Tabac_any==1 AND tamano_nodulo_mm_bin=='(17.90...   
4    0.034  Sexo_binaria==1 AND profundidad_min_pleura_mm_...   
5    0.029  Sexo_binaria==1 AND profundidad_min_pleura_mm_...   
6    0.029  Tabac_any==1 AND profundidad_min_pleura_mm_bin...   
7    0.028  profundidad_min_pleura_mm_bin=='(0.347, 0.5]' ...   
8    0.027  Sexo_binaria==1 AND Tabac_any==1 AND profundid...   
9    0.026                   Enfisema_any==1 AND Tabac_any==1   

             target  size_sg  size_dataset  positives_sg  positives_dataset  \
0  T: Neumotórax==1       31           210            18                 64   
1  T: Neumotórax==1       50           210            23                 64   

/mnt/homeGPU/mcribilles/conda_envs/sd/lib/python3.11/site-packages/pysubgroup/binary_target.py:356: RuntimeWarning: invalid value encountered in divide
  p_subgroup = np.divide(positives_subgroup, instances_subgroup)
